# Fine-tune screening model trên Google Colab (GPU T4)

Notebook này train model sàng lọc systematic review trên GPU miễn phí của Colab — nhanh hơn CPU local ~10–20 lần.

**Trước khi chạy:** vào menu `Runtime` → `Change runtime type` → chọn **T4 GPU** → Save.

Sau đó chạy lần lượt từng cell từ trên xuống (Shift+Enter).

In [ ]:
# 1. Kiểm tra GPU — phải thấy "Tesla T4". Nếu báo lỗi: chưa bật GPU ở Runtime settings.
!nvidia-smi

In [ ]:
# 2. Clone repo (dataset 8005 mẫu đã nằm sẵn trong repo)
!git clone https://github.com/thanhuy8888/-n_Systematic-Review-AI.git srai
%cd srai

In [ ]:
# 3. Cài thư viện còn thiếu (torch, sklearn, matplotlib có sẵn trên Colab)
!pip install -q transformers seaborn

## Thí nghiệm A — Công thức run 2 kéo dài (distil-biobert, 12 epochs, freeze 2)

Công thức tốt nhất hiện tại (run 2: 8 epochs đạt ROC-AUC 86.7%, recall 78.7%) nhưng val ROC-AUC vẫn đang tăng khi hết epoch — chạy 12 epochs để tìm đỉnh thật. Script tự lưu checkpoint epoch có val ROC-AUC cao nhất và dùng nó cho đánh giá cuối, nên không sợ overfit ở các epoch chót.

⏱️ Khoảng 10–15 phút trên T4.

In [ ]:
!python experiments/baselines/finetune_pubmedbert.py \
    --model nlpie/distil-biobert --epochs 12 --lr 3e-5 \
    --batch-size 16 --max-len 256 --freeze-layers 2

In [ ]:
# Đóng gói kết quả thí nghiệm A để tải về
!zip -r -q ket_qua_A_distil_12ep.zip \
    sr_core/screening_model/finetuned_pubmedbert \
    sr_core/screening_model/finetuned_meta.json \
    experiments/results/transformer_confusion_matrix.png \
    experiments/results/transformer_roc_curve.png \
    experiments/results/transformer_pr_curve.png \
    experiments/results/transformer_probability_distribution.png
from google.colab import files
files.download('ket_qua_A_distil_12ep.zip')

## Thí nghiệm B — PubMedBERT bản đầy đủ (12 layers, freeze 6)

Model lớn đúng tên đề tài, trần kết quả cao nhất. Trên CPU local mất 4–6 giờ, trên T4 chỉ ~25–40 phút.

⚠️ Cell này GHI ĐÈ kết quả thí nghiệm A trong thư mục — hãy chạy cell tải về của A trước.

In [ ]:
!python experiments/baselines/finetune_pubmedbert.py \
    --epochs 8 --lr 2e-5 \
    --batch-size 16 --max-len 256 --freeze-layers 6

In [ ]:
# Đóng gói kết quả thí nghiệm B để tải về
!zip -r -q ket_qua_B_pubmedbert_full.zip \
    sr_core/screening_model/finetuned_pubmedbert \
    sr_core/screening_model/finetuned_meta.json \
    experiments/results/transformer_confusion_matrix.png \
    experiments/results/transformer_roc_curve.png \
    experiments/results/transformer_pr_curve.png \
    experiments/results/transformer_probability_distribution.png
from google.colab import files
files.download('ket_qua_B_pubmedbert_full.zip')

## Đưa kết quả về máy local

1. Giải nén file zip đã tải về.
2. Chép `finetuned_pubmedbert/` và `finetuned_meta.json` đè vào `sr_core/screening_model/` trong project local — app sẽ tự dùng model mới.
3. Chép 4 file `.png` vào `experiments/results/` nếu muốn dùng biểu đồ mới cho báo cáo.

**Lưu ý về số liệu báo cáo:** kết quả train trên GPU sẽ *xấp xỉ* chứ không trùng từng chữ số với run CPU (khác kernel + mixed precision). Chỉ thay số chính thức trong Bảng 5 nếu model mới **vượt rõ rệt** run 2 hiện tại (ROC-AUC 86.7%, recall 78.7%, F1 67.1%); nếu chỉ ngang ngửa thì giữ run 2 làm số chính thức để khỏi phải cập nhật lại toàn bộ báo cáo.